<a href="https://colab.research.google.com/github/solasobambo-prog/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/solasobambo-prog/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Ranking / Scoring**

My lane (Lane 4) asks "which pages should an editor review first?" That's a "which
ones first" question, not a yes/no question, and not a "what groups exist" question.
Per the framing-ml-problems task-type table, that maps to ranking/scoring, where the
output is a priority score per page, not a class label.

**Important distinction, what I am NOT building:**
- I am **not predicting a future outcome**. Unlike `is_declining_label` (built from
  `trend_direction`/`trend_pct`, a change over time), my score uses only the page's
  current state: this period's CTR vs. this period's position tier. No future window
  needed.
- I am **not rebuilding an existing product flag**. There is no "under-capturing"
  label already sitting in the dataset for me to reproduce. If there were, training
  a model on it would just relearn FlyRank's existing rule, not add anything new.

Instead, I'm **constructing a new proxy metric** from data that's already observed:
something like `ctr_gap = actual_ctr minus tier_average_ctr`. Both halves of that gap
exist right now, so this is grounded in observed data, not invented from nowhere, and
not a future prediction. Since there's no ground truth "this really was an opportunity"
label to check against, my success metric (Section 3) will focus on whether the ranking
surfaces real, trustworthy variation, not precision@K against a true label the way
notebook 02's decision tree was scored.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

**What I would predict/score:** a CTR gap per page, calculated as:

`ctr_gap = actual_ctr minus tier_average_ctr`

Where `tier_average_ctr` is the mean CTR for all pages in that page's own `position_tier`
(restricted to pages with enough impressions to trust the number, impressions_90d >= 100,
as established in ML-02).

**Where does this come from, observed outcome or defined rule?**

Both halves of the gap are observed values already in the dataset (`ctr` and
`position_tier`), so nothing is invented. But the gap itself is a **proxy I am
constructing**, not an existing label FlyRank stored for me (there is no
`is_underperforming` column). This matters because of the label trap
framing-ml-problems warns about: if I had instead tried to predict some existing
flag, I would just be learning FlyRank's rule back, not producing anything new.

A large negative `ctr_gap` (actual CTR well below tier average) is the proxy for
"this page may be under capturing clicks for its position." A gap near zero or
positive means the page is performing about as expected or better.

**Honest limitation:** since tier average CTR is itself a distributional midpoint,
roughly half of all pages will always sit below it by definition. So `ctr_gap` alone
is a starting proxy, not proof of a real content problem. Volume (from ML-02, sample
sizes as small as 1 to 5 rows in some tier and content type combinations) still needs
to be factored in before treating a gap as trustworthy.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

**Metric: percentage of the top-K ranked pages (by ctr_gap) that have impressions_90d >= 100**

Since Lane 4 has no ground truth "this page really was an opportunity" label, I cannot
use precision@K against a true label the way notebook 02 scored the decision tree
against `is_declining_label`. So instead of scoring accuracy, I am scoring trustworthiness
of the ranking itself.

**Why this metric:** ML-01 showed that small sample sizes (as low as 1 to 5 rows in some
tier and content type groups) can produce misleading or even nonsensical CTR values. If
my ranking's top candidates are dominated by low-impression pages, the whole ranking is
noise, not signal, no matter how large the gaps look. A high score on this metric means
the pages surfaced at the top are backed by enough real traffic to trust the number,
which is a precondition for the ranking being useful at all.

**What good looks like:** I would want the large majority (for example 90% or more) of
the top 50 ranked pages to meet the impressions_90d >= 100 threshold. If that number is
low, it tells me my ranking needs an explicit volume filter or weighting, not just a raw
gap sort.

**Computable today:** yes, this can be checked right now on the plain rule based ranking
(sorting by ctr_gap), before any model is involved. That gives me an honest baseline to
compare against later, per framing-ml-problems' instruction to name and compute the
metric before training anything.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

**One row = one content page.**

Each row is uniquely identified by `content_id` (a pseudonymized page identifier, used
only for grouping and joining, never as a feature, per the flyrank-data skill). The
dataframe above shows the Lane 4 slice: 22,006 pages out of 30,000 total, filtered to
those with impressions_90d >= 100, the same volume threshold established in ML-02.

Each row carries everything needed to compute the target proxy from Section 2:
- `position_tier`: the page's current ranking tier
- `ctr`: the page's own observed CTR
- `tier_avg_ctr`: the mean CTR for all pages in that same tier
- `ctr_gap`: `ctr` minus `tier_avg_ctr`, the proxy for under or over capturing clicks

For example, row 10 (`content_d8ee6cc6d642`) sits in `top_3` with a CTR of 1.55% against
a tier average of 0.33%, a strongly positive gap of about +1.22, suggesting it is
capturing far more clicks than expected for its position. By contrast, row 5
(`content_d4084a4bc775`) sits in `page_1` with a CTR of just 0.03% against a tier average
of 0.35%, a gap of about minus 0.32, the kind of page my ranking should surface as a
review candidate.

This confirms the unit of analysis is a single page's current performance, not a client,
not a day, and not an aggregated group.

In [8]:
!git clone https://github.com/solasobambo-prog/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 147 (delta 54), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 1.87 MiB | 15.59 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/flyrank-ml-internship/flyrank-ml-internship


In [9]:
# Section 4: Unit of analysis — one row = one page
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Build the Lane 4 slice: pages with enough impressions to trust their CTR
lane4_df = df[df["impressions_90d"] >= 100].copy()

# Compute tier average CTR and the ctr_gap proxy from Section 2
tier_avg_ctr = lane4_df.groupby("position_tier")["ctr"].transform("mean")
lane4_df["tier_avg_ctr"] = tier_avg_ctr
lane4_df["ctr_gap"] = lane4_df["ctr"] - lane4_df["tier_avg_ctr"]

print(f"Unit of analysis: one row = one content page")
print(f"Lane 4 slice size: {len(lane4_df):,} pages (of {len(df):,} total)")
print(f"\nSample rows:")
lane4_df[["content_id", "position_tier", "ctr", "tier_avg_ctr", "ctr_gap", "impressions_90d"]].head(10)

Unit of analysis: one row = one content page
Lane 4 slice size: 22,006 pages (of 30,000 total)

Sample rows:


,content_id,position_tier,ctr,tier_avg_ctr,ctr_gap,impressions_90d
0,content_304f48230142,striking,0.76,0.255782,0.504218,3803
1,content_a1fb4e703a9e,page_3_5,0.05,0.142359,-0.092359,15320
2,content_9aa793d4d895,page_3_5,0.09,0.142359,-0.052359,12581
3,content_331d6c4de07b,page_1,0.49,0.354760,0.135240,11751
4,content_d99b7a2d90ca,page_3_5,0.13,0.142359,-0.012359,19140
5,content_d4084a4bc775,page_1,0.03,0.354760,-0.324760,3970
7,content_a63219c6e95a,page_3_5,0.06,0.142359,-0.082359,1724
8,content_5e6c160719bc,page_3_5,0.09,0.142359,-0.052359,32574
9,content_c27558df2b0c,page_1,0.16,0.354760,-0.194760,1240
10,content_d8ee6cc6d642,top_3,1.55,0.334128,1.215872,20919


## 5. Why ML beats a fixed rule here

**Honest starting point:** the `ctr_gap` proxy itself (Section 2) is not ML, it is a
plain rule (subtract the tier average from the actual CTR). Per framing-ml-problems,
"sometimes a plain rule is the right answer," and for a single signal like this, a
rule is genuinely enough. I do not want to force ML where it is not earning its place.

**Where a fixed rule starts to break down:** a single-signal gap treats every
underperforming page as equally worth reviewing, but that is not true. From ML-01, I
already saw that:
- Small sample sizes (as low as 1 to 5 pages in some tier and content_type groups) can
  produce a large-looking gap that is really just noise, not a real pattern.
- The best next step is not just "sort by gap," but weigh the gap against how much
  volume backs it, and possibly other signals like `content_age_days`, `word_count`,
  `days_since_last_update`, or `content_type`, which may explain why some pages
  underperform their tier and others do not.

Writing a single if-statement that correctly balances gap size, sample volume, content
age, and content type all at once, and does so consistently across thousands of pages
with different combinations, becomes hard to hand-write and maintain as more signals
are added. This is the exact situation framing-ml-problems describes as ML earning its
place: "the pattern is real but too messy to write by hand, many signals, tangled,
shifting over time."

**Where this leaves me:** my baseline for Lane 4 will likely start as the plain rule
(`ctr_gap`, volume filtered), since that is honest and defensible today. ML's role
comes in later, if combining multiple signals into one priority score meaningfully
outperforms the plain rule, which I will need to actually test and show, not assume.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.